# Action tools and task state

Everything in notebook 6 was **read-only**. `lookup_order` reads `orders.json`;
`process_return_request` computes a `decision` string. Nothing is ever written, so
the assistant can only *describe* outcomes — which is exactly how it ended up
telling a customer "I'm escalating your case to a specialist now" when no such
thing existed or happened.

This notebook adds two tools with real side effects:

- `create_return_authorization` — writes an RMA record to `returns.json`
- `escalate_to_human` — writes a ticket to `escalations.json`

and a small **case state** that those tools mutate. That gives the turn a
*terminal condition*: the loop now ends because the task reached a final state,
not because Claude ran out of things to say.

Everything else — the extraction tool, the business rules, schema and semantic
validation, the retry-on-invalid-input logic — carries over from notebook 6
unchanged.

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"


## Tool definitions

The two read tools are unchanged. The two new ones are marked `strict: True` —
these calls *write to disk*, so we want the API to guarantee the shape before we
ever look at it, and then validate the semantics ourselves on top.

In [2]:
tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its order ID and return its item, status, order date, and total. Call this whenever the customer references an order number or asks about the state of an existing order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, e.g. A1001"
                }
            },
            "required": ["order_id"]
        }
    },
    {
        # Structured output demo: this tool's only job is to shape Claude's
        # output. We mark it `strict: True` so the API guarantees the
        # returned input matches the schema exactly — every field present,
        # enums honored, no extra keys — no parsing/regex needed on our side.
        "name": "extract_return_request",
        "description": "Call this whenever a customer describes a return or refund request, to capture it as structured data before deciding what to do next. Fill in every field as best you can from the conversation; use null/'unclear' and list missing_information for anything not yet stated.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID mentioned by the customer, e.g. A1001, or null if they haven't given one."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "billing_dispute",
                        "late_delivery",
                        "unclear",
                        "other"
                    ],
                    "description": "The customer's stated reason for the return/refund. Use 'unclear' if not stated clearly enough to classify."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgent the request sounds, based on tone and content (e.g. angry, time-sensitive)."
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["order_id", "reason", "item_condition", "preferred_resolution"]
                    },
                    "description": "Which pieces of information the customer has not yet provided."
                },
                "summary": {
                    "type": "string",
                    "description": "One-sentence, neutral summary of what the customer wants."
                }
            },
            "required": ["order_id", "reason", "urgency", "missing_information", "summary"],
            "additionalProperties": False
        }
    },
    {
        # ACTION TOOL. Unlike everything above, calling this changes the world:
        # it writes an RMA record to returns.json and closes the case. Only call
        # it once the rules have actually approved the return.
        "name": "create_return_authorization",
        "description": "Authorize a return and issue an RMA number. Call this ONLY after extract_return_request has returned decision 'approve_return'. This creates a real record — do not call it speculatively, and do not tell the customer their return is approved until this tool has returned ok: true.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID being returned, e.g. A1001."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "late_delivery",
                        "other"
                    ],
                    "description": "The approved return reason, copied from the extract_return_request call."
                },
                "note": {
                    "type": "string",
                    "description": "One-sentence note for the warehouse team about the condition or context of the return."
                }
            },
            "required": ["order_id", "reason", "note"],
            "additionalProperties": False
        }
    },
    {
        # ACTION TOOL. Files a ticket for a human specialist and closes the case.
        "name": "escalate_to_human",
        "description": "File a ticket for a human specialist and hand the case off. Call this ONLY after extract_return_request has returned decision 'escalate_to_specialist', or when you genuinely cannot resolve the request with the tools available. Do not tell the customer you are escalating until this tool has returned ok: true.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID if known, or null if the customer never gave a valid one."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "billing_dispute",
                        "high_urgency",
                        "order_not_found",
                        "policy_exception",
                        "other"
                    ],
                    "description": "Why this needs a human."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgently a human should pick this up."
                },
                "summary": {
                    "type": "string",
                    "description": "Neutral summary of the situation for the specialist who picks this up, including anything the customer told you that isn't in the order record (e.g. an email address)."
                }
            },
            "required": ["order_id", "reason", "urgency", "summary"],
            "additionalProperties": False
        }
    }
]

## Case state — the thing the tools actually move

`case` is the task state for the current customer. Read tools don't touch it;
action tools do. The loop reads it to decide whether the task is finished.

This is the whole point of the notebook: once state exists, "done" is something
the *system* knows, not something we infer from Claude's `stop_reason`.

In [3]:
import json
import uuid
from datetime import date
from pathlib import Path

ORDERS_FILE = Path("orders.json")
RETURNS_FILE = Path("returns.json")
ESCALATIONS_FILE = Path("escalations.json")

# Once the case reaches one of these, the task is over — a return was
# authorized, or a human owns it now. Nothing Claude says afterwards reopens it.
TERMINAL_STATES = {"return_authorized", "escalated"}

# You can only return something you actually received. A cancelled order was
# never shipped, so a return against one is either a mix-up or an attempt to
# get a refund for goods that never left the warehouse — either way an RMA is
# the wrong answer. Status is a fact about the order, so this is a rule, not
# something to leave to the model's judgement.
NON_RETURNABLE_STATUSES = {"cancelled"}


def new_case() -> dict:
    """Fresh task state for one customer conversation.

    `last_decision` is the most recent result from the business rules. The
    action tools are gated on it — see validate_action_semantics.
    """
    return {"state": "open", "actions": [], "last_decision": None}


case = new_case()


def find_order(order_id: str) -> dict | None:
    """Look up one order in orders.json, case-insensitively."""
    orders = json.loads(ORDERS_FILE.read_text())
    return next((o for o in orders if o["order_id"].lower() == order_id.lower()), None)


def append_record(path: Path, record: dict) -> None:
    """Append a record to a JSON array file, creating it if it doesn't exist."""
    records = json.loads(path.read_text()) if path.exists() else []
    records.append(record)
    path.write_text(json.dumps(records, indent=2) + "\n")

## Tool implementations

`lookup_order` and `process_return_request` are as they were, plus one new rule:
an order whose status is in `NON_RETURNABLE_STATUSES` gets decision
`not_returnable`. Without it, a customer could ask to return a **cancelled**
order, get asked "why?", answer anything at all, and walk away with an RMA and a
refund for goods that were never shipped. Eligibility is a property of the order,
so it's checked before we ask the customer anything.

The two action tools are new, and both follow the same shape:

1. re-check the precondition themselves (never trust the caller — including Claude),
2. guard against duplicates, because a retrying model must not be able to open two RMAs,
3. write the record,
4. flip `case["state"]`,
5. return a JSON result with `ok` and an ID Claude can quote back to the customer.

Step 1 matters: the tool is the last line of defense, not the prompt. So the
cancelled-order rule lands in three places — the rules that decide, the validator
that gates the call, and the tool that writes the file. That's not redundancy for
its own sake; each one is the only thing standing there if the layer above it is
bypassed.

In [4]:
def lookup_order(order_id: str) -> str:
    """Look up an order by ID in orders.json and return its details as JSON, or an error message."""
    order = find_order(order_id)
    if order is None:
        return f"Error: no order found with ID '{order_id}'."

    return json.dumps(order)


def process_return_request(order_id, reason, urgency, missing_information, summary) -> str:
    """Apply business rules to an extracted return request and return a JSON
    result describing what should happen next — Claude reads this to decide
    which action tool (if any) to call."""
    def record(decision: str, **extra) -> str:
        """Remember what the rules decided. The action tools are gated on this,
        so a decision that was never made can't be acted on."""
        case["last_decision"] = {"decision": decision, "order_id": order_id}
        return json.dumps({"decision": decision, **extra})

    if not order_id or "order_id" in missing_information:
        return record("need_order_id")

    order = find_order(order_id)
    if order is None:
        return record("order_not_found", order_id=order_id)

    if reason == "billing_dispute" or urgency == "high":
        # Checked first, and deliberately so: "you charged me for an order you
        # cancelled" is a real problem that a human needs to look at, even
        # though the order itself isn't returnable.
        decision = "escalate_to_specialist"
    elif order["status"] in NON_RETURNABLE_STATUSES:
        # Ahead of need_clarification — there's no reason the customer could
        # give that would make a cancelled order returnable, so asking them
        # for one just wastes their time and ends in a refusal anyway.
        decision = "not_returnable"
    elif reason == "unclear":
        decision = "need_clarification"
    else:
        decision = "approve_return"

    return record(
        decision,
        order=order,
        reason=reason,
        urgency=urgency,
        summary=summary,
    )


def create_return_authorization(order_id: str, reason: str, note: str) -> str:
    """ACTION: write an RMA record to returns.json and close the case."""
    # The prompt says only to call this after an approve_return decision, but a
    # prompt isn't an enforcement mechanism — re-check the order ourselves.
    order = find_order(order_id)
    if order is None:
        return json.dumps({
            "ok": False,
            "error": f"No order '{order_id}' exists — cannot authorize a return for it.",
        })

    # Same reasoning: the rules already refuse this, and so does the validator,
    # but the tool is the last thing standing between a bad call and a refund
    # record on disk.
    if order["status"] in NON_RETURNABLE_STATUSES:
        return json.dumps({
            "ok": False,
            "error": (
                f"Order '{order['order_id']}' is {order['status']} — it was never "
                "delivered, so there is nothing to return and no RMA was created."
            ),
        })

    # Idempotency: if the model retries or gets confused, it must not be able to
    # open a second RMA for the same case.
    existing = next(
        (a for a in case["actions"] if a["tool"] == "create_return_authorization"),
        None,
    )
    if existing:
        return json.dumps({
            "ok": True,
            "rma_id": existing["id"],
            "note": "This case already has an authorized return; returning the existing RMA.",
        })

    rma_id = f"RMA-{uuid.uuid4().hex[:8].upper()}"
    append_record(RETURNS_FILE, {
        "rma_id": rma_id,
        "order_id": order["order_id"],
        "item": order["item"],
        "refund_amount": order["total"],
        "reason": reason,
        "note": note,
        "created_at": date.today().isoformat(),
    })

    case["state"] = "return_authorized"
    case["actions"].append({"tool": "create_return_authorization", "id": rma_id})

    return json.dumps({
        "ok": True,
        "rma_id": rma_id,
        "item": order["item"],
        "refund_amount": order["total"],
        "ship_back_within_days": 14,
    })


def escalate_to_human(order_id, reason: str, urgency: str, summary: str) -> str:
    """ACTION: file a specialist ticket in escalations.json and close the case."""
    existing = next(
        (a for a in case["actions"] if a["tool"] == "escalate_to_human"),
        None,
    )
    if existing:
        return json.dumps({
            "ok": True,
            "ticket_id": existing["id"],
            "note": "This case is already escalated; returning the existing ticket.",
        })

    ticket_id = f"ESC-{uuid.uuid4().hex[:6].upper()}"
    append_record(ESCALATIONS_FILE, {
        "ticket_id": ticket_id,
        "order_id": order_id,
        "reason": reason,
        "urgency": urgency,
        "summary": summary,
        "created_at": date.today().isoformat(),
    })

    case["state"] = "escalated"
    case["actions"].append({"tool": "escalate_to_human", "id": ticket_id})

    return json.dumps({
        "ok": True,
        "ticket_id": ticket_id,
        "response_within_hours": 4 if urgency == "high" else 24,
    })


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool_use block to its implementation."""
    if tool_name == "lookup_order":
        return lookup_order(tool_input["order_id"])

    if tool_name == "extract_return_request":
        return process_return_request(**tool_input)

    if tool_name == "create_return_authorization":
        return create_return_authorization(**tool_input)

    if tool_name == "escalate_to_human":
        return escalate_to_human(**tool_input)

    return f"Error: unknown tool '{tool_name}'."

## Validation and retry

Same two layers as notebook 6 — schema first, then the semantic checks a schema
can't express — now extended to the action tools. The stakes are higher here: a
bad `extract_return_request` call produces a wrong reply, but a bad
`create_return_authorization` call writes a wrong record to disk.

In [5]:
import re
from jsonschema import Draft7Validator

# One schema validator per tool, keyed by name, built once from `tools`.
_SCHEMA_VALIDATORS = {
    tool["name"]: Draft7Validator(tool["input_schema"]) for tool in tools
}

_ORDER_ID_RE = re.compile(r"^[A-Za-z]\d+$")


def validate_schema(tool_name: str, tool_input: dict) -> list[str]:
    """Validate tool_input against the tool's own JSON schema. Returns a list of
    human-readable error strings; empty means valid."""
    validator = _SCHEMA_VALIDATORS.get(tool_name)
    if validator is None:
        return [f"Unknown tool '{tool_name}'."]

    return [error.message for error in validator.iter_errors(tool_input)]


def validate_return_request_semantics(tool_input: dict) -> list[str]:
    """Business-rule checks the JSON schema can't express for extract_return_request."""
    errors = []

    order_id = tool_input.get("order_id")
    missing = tool_input.get("missing_information", [])
    summary = tool_input.get("summary", "")

    if order_id is not None:
        if not _ORDER_ID_RE.match(order_id.strip()):
            errors.append(
                f"order_id '{order_id}' doesn't look like a valid order ID "
                "(expected a letter followed by digits, e.g. A1001)."
            )
        if "order_id" in missing:
            errors.append(
                "order_id is set but 'order_id' is also listed in missing_information — "
                "these are inconsistent."
            )
    elif "order_id" not in missing:
        errors.append("order_id is null but 'order_id' is not listed in missing_information.")

    if not summary.strip():
        errors.append("summary is empty — provide a one-sentence neutral summary.")
    elif len(summary.split()) < 3:
        errors.append("summary is too short to be a meaningful one-sentence summary.")

    return errors


def validate_action_semantics(tool_name: str, tool_input: dict) -> list[str]:
    """Checks that apply to the two write tools. These run before anything
    touches disk, and they also stop the model from acting on a case that is
    already closed."""
    errors = []

    if case["state"] in TERMINAL_STATES:
        errors.append(
            f"This case is already in state '{case['state']}' — it's finished. "
            "Don't take another action on it."
        )

    order_id = tool_input.get("order_id")
    decision = (case["last_decision"] or {}).get("decision")

    if tool_name == "create_return_authorization":
        # The prompt says to call this only after an approve_return decision.
        # This is what actually enforces it: without the gate, a model that
        # skipped or botched the extraction step can still write an RMA.
        if decision != "approve_return":
            errors.append(
                f"The rules haven't approved this return (last decision: {decision or 'none'}). "
                "Call extract_return_request and follow its decision before authorizing anything."
            )

        # An RMA without a real order is a record nobody can act on.
        order = None
        if not _ORDER_ID_RE.match((order_id or "").strip()):
            errors.append(
                f"order_id '{order_id}' doesn't look like a valid order ID "
                "(expected a letter followed by digits, e.g. A1001)."
            )
        else:
            order = find_order(order_id)
            if order is None:
                errors.append(
                    f"No order '{order_id}' exists — ask the customer to re-check "
                    "their order ID instead of authorizing a return."
                )

        # Belt and braces with the rules: the decision gate above already blocks
        # this, but only as long as last_decision refers to the same order. Check
        # the order in *this* call rather than trusting that they match.
        if order is not None and order["status"] in NON_RETURNABLE_STATUSES:
            errors.append(
                f"Order '{order['order_id']}' is {order['status']} — a cancelled order "
                "was never delivered, so it can't be returned. Explain that to the "
                "customer; if they say they were charged for it, treat that as a "
                "billing_dispute and re-run extract_return_request."
            )

    if tool_name == "escalate_to_human":
        # Two legitimate routes to a human: the rules said so, or the customer's
        # order ID doesn't exist and we've run out of ways to help them.
        if decision not in ("escalate_to_specialist", "order_not_found") \
                and tool_input.get("reason") != "order_not_found":
            errors.append(
                f"The rules haven't called for an escalation (last decision: {decision or 'none'}). "
                "Call extract_return_request and follow its decision."
            )

        # order_id is nullable here (we escalate cases where it was never found),
        # but if one is given it has to be real.
        if order_id is not None and find_order(order_id) is None:
            errors.append(
                f"order_id '{order_id}' doesn't match any order — pass null "
                "instead and explain the situation in the summary."
            )
        if len(tool_input.get("summary", "").split()) < 5:
            errors.append(
                "summary is too thin — a human is going to read this cold, so "
                "include what the customer wants and anything they told you."
            )

    return errors


def validate_tool_call(tool_name: str, tool_input: dict) -> list[str]:
    """Run schema validation, then (if that passes) tool-specific semantic validation."""
    errors = validate_schema(tool_name, tool_input)
    if errors:
        return errors

    if tool_name == "extract_return_request":
        return validate_return_request_semantics(tool_input)

    if tool_name in ("create_return_authorization", "escalate_to_human"):
        return validate_action_semantics(tool_name, tool_input)

    return []

In [6]:
def add_user_message(messages: list, text: str) -> None:
    """Append a user turn to the conversation."""
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages: list, text: str) -> None:
    """Append an assistant turn to the conversation."""
    messages.append({"role": "assistant", "content": text})

## The loop: every turn must end, and must end with words

Notebook 6's loop was `while True` with exactly one exit: Claude stops asking for
tools. That is not a guarantee — a model that keeps calling tools, or keeps
re-calling one we keep rejecting, loops forever and the customer sees nothing.
So this version has **four** ways out, and all of them return text:

| Exit | When |
| --- | --- |
| Claude stops calling tools | the normal case |
| An action tool closed the case | `case["state"]` is terminal |
| Validation gave up on a tool | the retry budget is spent |
| `MAX_ITERATIONS` reached | the backstop — nothing loops forever |

The last three all route through `finish()`, which makes one more call with
`tool_choice: {"type": "none"}`. Claude writes a closing message and *cannot*
take another action. That single mechanism is what makes the exits safe.

`finish()` also takes a **closing note** that gets appended to the system prompt
for that one call. Claude can't see the loop, so it doesn't know whether it's
signing off on a closed case or admitting it got stuck — and a model that doesn't
know assumes the conversation continues. That's how you end up with "your return
is approved… is there anything else I can help you with?" one line before the
chat loop exits. The note tells it which ending it's writing: `CLOSED_CASE_NOTE`
forbids the trailing question, `UNRESOLVED_NOTE` requires one.

Three smaller robustness fixes:

- **Validation failures no longer fall through.** In notebook 6, once the retry
  budget was spent the code logged the failure and executed the call anyway.
  Survivable when everything was read-only; not now. The tool never runs.
- **`text_of()` replaces `next(...)`.** A response can legitimately contain no
  text block — `next()` raises `StopIteration`, which reads as "the assistant
  never answered". Joining the text blocks always yields a string.
- **`max_tokens` is handled.** A truncated turn can hold a half-formed `tool_use`
  with no matching `tool_result`, which makes the *next* request invalid. We drop
  that turn rather than poisoning the conversation.

In [7]:
SYSTEM_PROMPT = """You are a helpful shop assistant for an online store.

If the customer is just asking about the status of an existing order, call
lookup_order directly.

If the customer describes wanting a return or refund, call
extract_return_request first — even if they haven't given you every detail
yet, since the tool's result will tell you what's missing and what to do
next. Fill in every field of that tool as best you can from the conversation
so far; don't ask the customer questions before calling it.

Follow the extract_return_request result's "decision" field to guide what you
do next:
- need_order_id: ask for their order ID. No action tool.
- order_not_found: let them know that order ID doesn't match anything, and ask
  them to double-check it. No action tool.
- need_clarification: ask why they'd like to return the item. No action tool.
- not_returnable: the order's status means it can't be returned — a cancelled
  order was never delivered, so there is nothing to send back. Say so plainly
  and kindly, without blaming them, and don't ask for a return reason. If they
  tell you they were charged for it anyway, that is a billing_dispute: re-run
  extract_return_request with that reason and follow the new decision. No
  action tool.
- escalate_to_specialist: call escalate_to_human, then tell the customer their
  ticket number and when to expect a reply.
- approve_return: call create_return_authorization, then confirm the return with
  the RMA number and explain next steps.

Two rules about the action tools (create_return_authorization and
escalate_to_human):

1. Never tell the customer that something has happened unless the corresponding
   tool has returned ok: true. Do not say you are escalating, have escalated, or
   have approved a return before the tool call. If a tool returns ok: false or
   an error, say plainly that it didn't go through — do not describe the failed
   action as if it succeeded.
2. Each case gets at most one action. Once a return is authorized or a case is
   escalated, the case is closed.

If the customer keeps insisting on an order ID that doesn't exist, and you've
already asked them to double-check it, escalate_to_human with reason
order_not_found rather than continuing to loop.
"""

# Appended to the system prompt for the final message when an action tool closed
# the case. Without it the model signs off with "is there anything else I can
# help you with?" — a question the customer can't answer, because the chat loop
# is about to exit. The closing message has to match what the system just did.
CLOSED_CASE_NOTE = """CLOSING MESSAGE. This case is now closed and this is the last thing you will
say — the conversation ends here and you will not see a reply. Write a short
sign-off that confirms what was done, quotes the RMA or ticket number, and says
what happens next and roughly when. Do not offer further help, do not ask
whether there's anything else, and do not ask any question at all. If they need
something more, they can start a new request — you may say that once, briefly.
"""

# The other exits: the case is still open, we just couldn't get any further this
# turn. Here a question is exactly right — we need something from the customer.
UNRESOLVED_NOTE = """CLOSING MESSAGE. You could not complete this request, and no action was taken —
no return was authorized and no ticket was filed. Say so honestly without
inventing a reason, and never imply anything happened. Tell the customer in one
or two sentences what you need from them to move forward, or ask them to try
again.
"""

MAX_VALIDATION_RETRIES = 2
# Backstop on the tool-use loop. Nothing here is allowed to run forever — if we
# hit this, we stop and say something rather than leaving the customer hanging.
MAX_ITERATIONS = 8

# The whole conversation lives in this list — we resend it on every call.
messages = []


def text_of(response) -> str:
    """Join a response's text blocks into one string.

    Never raises. A response can legitimately have no text block (a truncated
    turn, or a turn that was pure tool_use), and the caller still needs
    something to show the customer."""
    parts = [block.text for block in response.content if block.type == "text"]
    joined = "\n".join(parts).strip()
    return joined or f"(no text in response; stop_reason={response.stop_reason})"


def finish(messages: list, reason: str, closing_note: str) -> str:
    """End the turn with a closing message and no ability to act.

    Used for every exit except 'Claude stopped on its own': tool_choice "none"
    means Claude can only write text here, so this can't start a new action or
    a new tool-call cycle.

    `closing_note` is stapled onto the system prompt for this one call. The
    model has no way to know why the loop is ending — whether the case closed or
    we gave up — so we have to tell it, or the sign-off contradicts what the
    system actually did."""
    print(f"[turn ending: {reason}]")
    final = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM_PROMPT + "\n\n" + closing_note,
        tools=tools,
        tool_choice={"type": "none"},
        messages=messages,
    )
    messages.append({"role": "assistant", "content": final.content})
    return text_of(final)


def send_message(messages: list) -> str:
    """Call Claude with the current conversation, running the tool-use loop
    until one of the four exits in the table above. Always returns text."""
    # Counts validation failures across the whole turn (all tool calls), so a
    # model that keeps producing invalid input can't loop forever.
    validation_retries = 0

    for iteration in range(1, MAX_ITERATIONS + 1):
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "max_tokens":
            # A truncated turn can contain a half-formed tool_use block with no
            # tool_result to match it, which makes the NEXT request invalid.
            # Drop it instead of poisoning the conversation.
            print("[warning] response truncated at max_tokens — dropping that turn")
            return "Sorry — I ran out of room mid-reply. Could you send that again?"

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return text_of(response)

        tool_results = []
        gave_up = False
        for block in response.content:
            if block.type == "tool_use":
                print(f"[tool call] {block.name}({block.input})")
                errors = validate_tool_call(block.name, block.input)

                if errors:
                    validation_retries += 1
                    give_up = validation_retries > MAX_VALIDATION_RETRIES
                    gave_up = gave_up or give_up
                    label = "giving up" if give_up else f"retry {validation_retries}/{MAX_VALIDATION_RETRIES}"
                    print(f"[validation failed, {label}] {errors}")

                    error_text = "Your tool call was invalid:\n" + "\n".join(
                        f"- {error}" for error in errors
                    )
                    # Either way the tool does NOT run — an action tool writes to
                    # disk, so an exhausted retry budget must mean "don't", not
                    # "do it anyway".
                    error_text += (
                        "\nDon't call this tool again — explain the situation to the customer instead."
                        if give_up
                        else "\nPlease call the tool again with corrected input."
                    )
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": error_text,
                        "is_error": True,
                    })
                    continue

                result = execute_tool(block.name, block.input)
                if block.name in ("create_return_authorization", "escalate_to_human"):
                    print(f"[action] {block.name} -> {result}")

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        messages.append({"role": "user", "content": tool_results})

        # Exit 2: an action tool closed the case — the task is over.
        if case["state"] in TERMINAL_STATES:
            return finish(messages, f"case closed ({case['state']})", CLOSED_CASE_NOTE)

        # Exit 3: we've stopped executing this tool, so leaving the model free
        # to call it again is how you get an infinite retry cycle.
        if gave_up:
            return finish(messages, "validation retry budget exhausted", UNRESOLVED_NOTE)

    # Exit 4: backstop. Claude is still asking for tools after MAX_ITERATIONS
    # rounds — stop and hand back whatever it can say.
    return finish(messages, f"hit MAX_ITERATIONS={MAX_ITERATIONS}", UNRESOLVED_NOTE)

## Chat

The loop below stops once the case is closed — there's nothing left for the
assistant to do, and any further conversation belongs to whoever picks up the
ticket. Check `returns.json` / `escalations.json` afterwards: unlike notebook 6,
the outcome now exists somewhere other than the chat transcript.

Call `case.update(new_case())` (or restart the kernel) to start a fresh case.

In [8]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    # VS Code's input box returns "" for both Escape and a blank Enter, so
    # blank input doubles as the way to quit here.
    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    add_user_message(messages, user_input)
    # send_message appends the assistant turn(s) itself, including any
    # intermediate tool_use/tool_result turns from the tool loop.
    reply = send_message(messages)

    print(f"Assistant: {reply}")

    if case["state"] in TERMINAL_STATES:
        print(f"\n--- case closed ({case['state']}): {case['actions']} ---")
        break

User: Hello I want to return my order, that's the worng item I got
Assistant: I'd like to help you with that return! Let me get some information from you.

**Could you please provide your order ID?** It's usually a code like A1001 or similar, and you should find it in your confirmation email or order history.

Once I have that, I can look into the details and help you process the return for the wrong item.
User: A1005
[tool call] lookup_order({'order_id': 'A1005'})
Assistant: I've looked up your order A1005, and I can see it was cancelled. Unfortunately, since the order was cancelled, it was never delivered — so there's nothing to send back to us.

However, if you were charged for this order even though it was cancelled, that's a billing issue we should look into. **Were you charged for order A1005?**
User: Yes, I was double charged for that and I am mad about ot
[tool call] escalate_to_human({'order_id': 'A1005', 'reason': 'billing_dispute', 'urgency': 'high', 'summary': 'Customer rep